# Short-horizon forecasting of daily dengue hospital admissions in Dhaka, 2020–2024
### Final analysis notebook for the revised manuscript (all reviewer and editor responses)

Run all cells in order (**Runtime → Run all**). On a standard Colab CPU the full run takes roughly 1.5–2.5 hours; the slowest parts are the machine-learning tuning (Sections 5 and 8c) and the LSTM (Section 6). Every table and figure is saved to `out/` and downloaded as one zip at the end.

**Naming in outputs:** `orig` = training from 2020; `restr` = training from 2022 (excludes 2020–2021, which disagree with official DGHS totals). `P1` = Phase 1 (test year 2024, typical); `P2` = Phase 2 (test year 2023, outbreak).

| Section | Content | Manuscript output | Reviewer / editor comments |
|---|---|---|---|
| 1 | Descriptive statistics | Table 1, Fig 3, Fig 4 | Editor 2 |
| 2 | Data audit | Table S1, Fig S1 | R1-1, R2-1/2/5, R3-13, R4-8 |
| 3 | Stationarity and heteroskedasticity | Table 2 | R1-6 |
| 4 | SARIMA protocol: training-only transformation choice, order search, residual checks, naive benchmarks | Table 3, Table S2, Fig 5 | R1-3/6/7, R2-6/14, R3-8/10, R4-3 |
| 5 | Machine learning: original 2-feature models, hybrid gate, fair common feature set | Tables 4–5, Table S5, Fig S3 | R1-4, R3-3/6, R4-5 |
| 6 | LSTM under the common protocol | Table S9 | R3-6, R4-5 |
| 7 | Diebold–Mariano tests, bootstrap CIs, metrics by transmission period | Table S8 | R1-7, R2-7/8, R3-7, R4-7 |
| 8 | Weather: Granger tests, SARIMAX out of sample, ML with weather | Tables S3, S4, S10 | R1-2, R2-10, R3-4/5, R4-4 |
| 9 | Forecast horizons of 7, 14 and 28 days; same horizons with vs without weather | Tables S11–S12, Fig S2 | R4-2, R4-4 |
| 10 | 2027 scenario: Fourier order, hindcasts, prediction intervals | Tables S6–S7, Fig 6 | R1-5, R2-11, R3-11/12, R4-1 |
| 11 | Study-area map | Fig 1 | R1-8, R2-3 |
| 12 | Workflow diagram | Fig 2 | R2-15 |

*AI disclosure: this notebook's revision code was developed with assistance from Claude (Anthropic) and reviewed and run by the authors.*

## 0. Setup

In [ ]:
!pip install -q catboost statsmodels scikit-learn xgboost lightgbm geopandas

In [ ]:
import os, sys, json, warnings, itertools, urllib.request
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt, matplotlib.dates as mdates
from scipy import stats
os.makedirs('out', exist_ok=True); os.makedirs('geo', exist_ok=True)
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'svg.fonttype':'none'})
SEED = 42
DATA_URL = 'https://raw.githubusercontent.com/Sakther12/Dengue_weather_2020-2024/main/dhaka_dengue_weather_2020_2024.csv'
LOCAL_FILE = 'dhaka_dengue_weather_2020_2024.csv'   # fallback: upload the Kaggle file to the Colab session

def load():
    try: d = pd.read_csv(LOCAL_FILE)
    except FileNotFoundError: d = pd.read_csv(DATA_URL)
    d.columns = [c.replace('﻿','') for c in d.columns]
    d['Date'] = pd.to_datetime(d['Date'])
    return d.sort_values('Date').set_index('Date')

df = load()
print(df.shape, df.index.min().date(), 'to', df.index.max().date())
import statsmodels, sklearn, xgboost, lightgbm, catboost, scipy
print('Python', sys.version.split()[0], '| statsmodels', statsmodels.__version__, '| scikit-learn', sklearn.__version__,
      '| xgboost', xgboost.__version__, '| lightgbm', lightgbm.__version__, '| catboost', catboost.__version__,
      '| scipy', scipy.__version__, '| numpy', np.__version__, '| pandas', pd.__version__, '| matplotlib', matplotlib.__version__)

Helper functions used throughout: SARIMA fitting, rolling one-day-ahead forecasts with parameters fixed after training, accuracy metrics, the Diebold–Mariano test, block-bootstrap confidence intervals and journal-style P values.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def fit_sarima(ytr, order=(2,1,2), sorder=(2,0,2,7), exog=None):
    m = SARIMAX(ytr, exog=exog, order=order, seasonal_order=sorder,
                enforce_stationarity=False, enforce_invertibility=False)
    return m.fit(disp=False, maxiter=200)

def onestep(res, ytest, exog_test=None):
    """Rolling one-step-ahead forecasts with parameters fixed after training."""
    full = res.append(ytest, exog=exog_test, refit=False)
    return full.predict(start=ytest.index[0], end=ytest.index[-1], dynamic=False)

def metrics(y, p, ytrain=None):
    y = np.asarray(y, float); p = np.clip(np.asarray(p, float), 0, None); e = y - p
    d = np.abs(y) + np.abs(p)
    out = dict(MAE=np.mean(np.abs(e)), RMSE=np.sqrt(np.mean(e**2)),
               sMAPE=np.mean(np.where(d == 0, 0, 2*np.abs(e)/np.where(d == 0, 1, d)))*100,
               R2=1 - np.sum(e**2)/np.sum((y - y.mean())**2))
    if ytrain is not None:
        out['MASE'] = out['MAE']/np.mean(np.abs(np.diff(np.asarray(ytrain, float))))
    return out

def dm_test(e1, e2, h=1, loss='abs'):
    """Diebold-Mariano with Harvey-Leybourne-Newbold correction; Bartlett HAC variance, lag max(h-1, 7). Negative = model 1 better."""
    d = (np.abs(e1) - np.abs(e2)) if loss == 'abs' else (e1**2 - e2**2)
    T = len(d); lag = max(h-1, 7); dc = d - d.mean()
    g = [np.sum(dc[k:]*dc[:T-k])/T for k in range(lag+1)]
    lrv = g[0] + 2*sum((1 - k/(lag+1))*g[k] for k in range(1, lag+1))
    s = d.mean()/np.sqrt(lrv/T) * np.sqrt((T + 1 - 2*h + h*(h-1)/T)/T)
    return s, 2*stats.t.sf(abs(s), T-1)

def boot_ci(e1, e2, B=2000, block=14, seed=1):
    """95% moving-block bootstrap CI for MAE(model 1) - MAE(model 2)."""
    rng = np.random.default_rng(seed); d = np.abs(e1) - np.abs(e2); T = len(d); nb = int(np.ceil(T/block)); out = []
    for _ in range(B):
        st = rng.integers(0, T-block+1, nb); idx = np.concatenate([np.arange(s, s+block) for s in st])[:T]; out.append(d[idx].mean())
    return np.percentile(out, [2.5, 97.5])

def pfmt(p):
    """Journal P-value style."""
    if p < .001: return 'P<.001'
    if p < .01:  return f'P={p:.3f}'.replace('0.', '.')
    if p > .99:  return 'P>.99'
    return f'P={p:.2f}'.replace('0.', '.')

def split_years(window, phase):
    """window: 'orig' (training from 2020) or 'restr' (from 2022); phase: 'P1' (test 2024) or 'P2' (test 2023)."""
    start = '2020' if window == 'orig' else '2022'
    return (start, '2023', '2024') if phase == 'P1' else (start, '2022', '2023')

## 1. Descriptive statistics (Table 1, Figures 3–4)
Skewed variables are summarised as median (IQR), approximately symmetric ones as mean (SD) (editor comment 2).

In [ ]:
y = df['Dengue_Cases'].astype(float)
rows = []
for label, col, kind in [('Daily dengue admissions','Dengue_Cases','median'), ('Temperature (°C)','Temperature','mean'),
                         ('Relative humidity (%)','Humidity','mean'), ('Daily rainfall (mm)','Rainfall_mm','median')]:
    s = df[col].astype(float)
    summ = (f'{s.median():.0f} ({s.quantile(.25):.0f}–{s.quantile(.75):.0f})' if kind == 'median'
            else f'{s.mean():.1f} ({s.std():.1f})')
    rows.append({'Variable (n = 1,826 days)': label, 'Summary': 'Median (IQR)' if kind == 'median' else 'Mean (SD)',
                 'Value': summ, 'Range': f'{s.min():g}–{s.max():g}', 'Skewness': round(s.skew(), 2)})
table1 = pd.DataFrame(rows); table1.to_csv('out/Table1_descriptive.csv', index=False); display(table1)

# Figure 3: daily series
fig, ax = plt.subplots(figsize=(11, 4))
ax.axvspan(pd.Timestamp('2020-01-01'), pd.Timestamp('2021-12-31'), color='#f2c4c4', alpha=.5,
           label='2020–2021: counts inconsistent with official DGHS totals')
ax.plot(y.index, y, color='#1f3b57', lw=.7, label='Daily reported dengue admissions')
ax.set_ylabel('Daily admissions'); ax.set_xlabel('Year'); ax.legend(loc='upper left', frameon=False, fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.savefig('out/Fig3_daily_admissions.png', dpi=600); plt.savefig('out/Fig3_daily_admissions.svg'); plt.show()

# Figure 4: monthly heatmap
import seaborn as sns
pivot = y.groupby([y.index.year, y.index.month]).sum().unstack()
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=.5, linecolor='white',
            cbar_kws={'label': 'Monthly admissions'}, ax=ax, annot_kws={'fontsize': 8})
ax.set_xlabel('Month'); ax.set_ylabel('Year'); plt.yticks(rotation=0)
plt.tight_layout(); plt.savefig('out/Fig4_monthly_heatmap.png', dpi=600); plt.savefig('out/Fig4_monthly_heatmap.svg'); plt.show()
print('Annual totals:', y.groupby(y.index.year).sum().astype(int).to_dict())

## 2. Data audit (Table S1, Figure S1)
Missing days, zero-count days, isolated single-day spikes (≥100 admissions and ≥3× the median of the six surrounding days), and comparison with published DGHS totals.

In [ ]:
import numpy as np, pandas as pd
df = load(); y = df['Dengue_Cases']
print('rows', len(df), df.index.min().date(), df.index.max().date())
full = pd.date_range(df.index.min(), df.index.max())
print('missing calendar days:', len(full.difference(df.index)), list(full.difference(df.index))[:5])
print('days in 2024 present:', (df.index.year==2024).sum())
print('districts', df.District.unique())
# yearly
yr = y.groupby(y.index.year).agg(['sum','mean','median','max'])
yr['zero_days'] = (y==0).groupby(y.index.year).sum()
# isolated spikes: day >= 100 and >= 5x the median of the 3 days on each side
med = pd.concat([y.shift(k) for k in [-3,-2,-1,1,2,3]],axis=1).median(axis=1)
spike = (y>=100)&(y>=3*med.clip(lower=1))
yr['isolated_spikes'] = spike.groupby(y.index.year).sum()
# day-to-day jumps
print(yr)
print('total zero days', (y==0).sum(), 'total spikes', spike.sum())
print(y[spike].to_string())
# zero runs
z=(y==0).astype(int); runs=z.groupby((z!=z.shift()).cumsum()).agg(['first','size'])
print('longest zero runs', runs[runs['first']==1]['size'].sort_values().tail(5).tolist())
# monthly table
m = y.groupby([y.index.year, y.index.month]).sum().unstack()
print(m)
# zero days by month-year
print((y==0).groupby([y.index.year,y.index.month]).sum().unstack())
# day of week effect
print(y.groupby(y.index.dayofweek).mean().round(1))
yr.to_csv('out/audit_yearly.csv'); m.to_csv('out/audit_monthly.csv')
y[spike].to_csv('out/audit_spikes.csv')

In [ ]:
# Table S1: dataset totals vs published official figures
y = df['Dengue_Cases'].astype(float)
tableS1 = pd.DataFrame([
  ['2020, full year', int(y['2020'].sum()), '1,193', 'DGHS, cited in PLOS Negl Trop Dis 2022 (COVID-19 onslaught is masking the 2021 dengue outbreak in Dhaka)', 'Inconsistent'],
  ['2021, January–May', int(y['2021-01':'2021-05'].sum()), '103 (Dhaka city)', 'Same source', 'Inconsistent'],
  ['2021, full year', int(y['2021'].sum()), '28,429 (all Bangladesh)', 'DGHS, cited in Trop Dis Travel Med Vaccines 2025', 'Exceeds national total'],
  ['2023, full year', int(y['2023'].sum()), '≈109,200 (34% of 321,179)', 'DGHS year-end totals as reported by Outbreak News Today', 'Within 0.7%'],
  ['2022, full year', int(y['2022'].sum()), 'not publicly reported for Dhaka city', '—', '—'],
  ['2024, full year', int(y['2024'].sum()), 'not publicly reported for Dhaka city', '—', '—']],
  columns=['Period', 'Kaggle dataset', 'Official figure', 'Source', 'Agreement'])
tableS1.to_csv('out/TableS1_dataset_vs_DGHS.csv', index=False); display(tableS1)

In [ ]:
df=load(); y=df.Dengue_Cases
sp=pd.read_csv('out/audit_spikes.csv',index_col=0,parse_dates=True).iloc[:,0]
fig,ax=plt.subplots(2,1,figsize=(11,7.5),gridspec_kw={'height_ratios':[1.4,1]})
a=ax[0]; a.axvspan(pd.Timestamp('2020-01-01'),pd.Timestamp('2021-12-31'),color='#f2c4c4',alpha=.5,label='2020–2021: counts inconsistent with DGHS reports')
a.plot(y.index,y,color='#1f3b57',lw=.7,label='Daily reported dengue admissions (Kaggle dataset)')
z=y[y==0]; a.scatter(z.index,z,s=6,color='#888',zorder=3,label=f'Zero-count days (n={len(z)})')
a.scatter(sp.index,sp,s=40,facecolor='none',edgecolor='#c0392b',lw=1.5,zorder=4,label=f'Isolated single-day spikes (≥100 and ≥3× local median) (n={len(sp)})')
a.set_ylabel('Daily admissions'); a.legend(loc='upper left',fontsize=8.5,frameon=False); a.set_title('A. Daily series with data-quality flags',loc='left',fontsize=11)
a.xaxis.set_major_locator(mdates.YearLocator()); a.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
b=ax[1]
lab=['2020\n(full year)','2021\n(Jan–May)','2023\n(full year)']
ds=[int(y['2020'].sum()),int(y['2021-01':'2021-05'].sum()),int(y['2023'].sum())]
off=[1193,103,round(0.34*321179)]
offl=['DGHS: 1,193\n(national)','DGHS: 103\n(Dhaka city)','DGHS: ≈109,200\n(Dhaka city, 34% of 321,179)']
x=np.arange(3); b.bar(x-.18,ds,.36,color='#1f3b57',label='Kaggle dataset'); b.bar(x+.18,off,.36,color='#e08e45',label='Official DGHS figure')
for i in range(3):
    b.text(x[i]-.18,ds[i],f'{ds[i]:,}',ha='center',va='bottom',fontsize=8.5); b.text(x[i]+.18,off[i],offl[i],ha='center',va='bottom',fontsize=7.5)
b.set_yscale('log'); b.set_ylim(50,6e5); b.set_xticks(x); b.set_xticklabels(lab); b.set_ylabel('Admissions (log scale)')
b.legend(frameon=False,fontsize=8.5,loc='upper left'); b.set_title('B. Dataset totals vs official DGHS figures',loc='left',fontsize=11)
plt.tight_layout(); plt.savefig('out/FigS1_data_audit.png',dpi=600); plt.show()

## 3. Stationarity and heteroskedasticity (Table 2)
ADF and KPSS under both specifications, plus ARCH-LM. The results are inconclusive, so first differencing is applied (Reviewer 1, comment 6).

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import het_arch
y = df['Dengue_Cases'].astype(float)
rows = []
for reg, name in [('c','constant'), ('ct','constant + trend')]:
    a = adfuller(y, regression=reg); k = kpss(y, regression=reg, nlags='auto')
    rows.append({'Test': f'ADF ({name})', 'Null hypothesis': 'Unit root', 'Statistic': round(a[0], 2), 'P': pfmt(a[1])})
    rows.append({'Test': f'KPSS ({name})', 'Null hypothesis': 'Stationary', 'Statistic': round(k[0], 2),
                 'P': pfmt(k[1]) + (' (bounded: ≥.10)' if k[1] >= .1 else '')})
arch = het_arch(y.diff().dropna(), nlags=7)
rows.append({'Test': 'ARCH-LM, 7 lags (differenced series)', 'Null hypothesis': 'No conditional heteroskedasticity',
             'Statistic': round(arch[0], 1), 'P': pfmt(arch[1])})
table2 = pd.DataFrame(rows); table2.to_csv('out/Table2_diagnostics.csv', index=False); display(table2)
print('ADF and KPSS results are inconclusive, so first differencing (d = 1) is applied as a conservative choice.')

## 4. Corrected SARIMA protocol (Table 3, Table S2, Figure 5)
For each phase and data window: (1) choose raw counts vs log1p with Duan smearing on an inner-validation year **inside the training data**; (2) search 81 SARIMA orders by AIC on training data; (3) forecast the test year one day ahead with fixed parameters; (4) compare with persistence, seasonal-naive and 7-day-mean benchmarks; (5) Ljung–Box tests on model residuals. Takes about 10 minutes.

In [ ]:
"""Corrected SARIMA protocol for Reviewer 1 (#3, #6, #7). All choices use training data only."""
import json, itertools, sys, numpy as np, pandas as pd
from joblib import Parallel, delayed
from statsmodels.stats.diagnostic import acorr_ljungbox
df=load(); y=df['Dengue_Cases'].astype(float); y.index.freq='D'

def split(window, phase):
    s = '2020' if window=='orig' else '2022'
    if phase=='P1': return y[s:'2023'], y['2024']
    return y[s:'2022'], y['2023']

def inner(tr):
    n = 365 if len(tr) >= 730 else 184
    return tr.iloc[:-n], tr.iloc[-n:]

def fc(tr, te, tf, order=(2,1,2), sorder=(2,0,2,7)):
    """one-step forecasts on te from model fit on tr, transform tf in {raw, log}"""
    if tf=='raw':
        res=fit_sarima(tr,order,sorder); return np.clip(onestep(res,te),0,None), res
    res=fit_sarima(np.log1p(tr),order,sorder)
    resid=res.resid.iloc[max(order[1],1)+14:]          # drop burn-in
    S=np.mean(np.exp(resid))                           # Duan smearing factor
    p=np.exp(onestep(res,np.log1p(te)))*S-1
    return np.clip(p,0,None), res

def aic_fit(tr, tf, o, so):
    try:
        r=fit_sarima(np.log1p(tr) if tf=='log' else tr, o, so); return (o,so,r.aic)
    except Exception as e: return (o,so,np.inf)

out={}
for window in ['orig','restr']:
  for phase in ['P1','P2']:
    key=f'{window}_{phase}'; tr,te=split(window,phase); itr,iva=inner(tr)
    # Step 1: transformation chosen on inner validation (prespecified model)
    tsel={tf: metrics(iva, fc(itr,iva,tf)[0])['MAE'] for tf in ['raw','log']}
    tf=min(tsel,key=tsel.get)
    # Step 2: order grid search by AIC on full training set (chosen transform)
    grid=[((p,1,q),(P,0,Q,7)) for p,q,P,Q in itertools.product(range(3),range(3),range(3),range(3))]
    aics=Parallel(n_jobs=-1)(delayed(aic_fit)(tr,tf,o,so) for o,so in grid)
    aics=sorted(aics,key=lambda t:t[2]); best=aics[0]
    # Step 3: test-set forecasts (prespecified 2,1,2x2,0,2 and grid-selected), both transforms for sensitivity
    preds={}
    for t2 in ['raw','log']:
        preds[f'pre_{t2}'],_=fc(tr,te,t2)
    preds['grid'],res=fc(tr,te,tf,best[0],best[1])
    # baselines
    preds['persistence']=y.shift(1)[te.index].values
    preds['seasonal_naive']=y.shift(7)[te.index].values
    preds['mean7']=y.shift(1).rolling(7).mean()[te.index].values
    # residual diagnostics on training fit
    r=res.resid.iloc[30:]; r=r/r.std()
    lb=acorr_ljungbox(r,lags=[7,14,21,28],return_df=True)
    lb_test=acorr_ljungbox(te.values-preds['grid'],lags=[7,14,21,28],return_df=True)
    pd.DataFrame(preds,index=te.index).assign(actual=te.values).to_csv(f'out/preds_{key}.csv')
    out[key]=dict(n_train=len(tr),inner_train=[str(itr.index[0].date()),str(itr.index[-1].date())],
        inner_val=[str(iva.index[0].date()),str(iva.index[-1].date())], inner_MAE=tsel, chosen_tf=tf,
        top5_aic=[(str(a),str(b),round(c,1)) for a,b,c in aics[:5]], n_grid=len(grid),
        best=str(best[:2]), lb_train=lb.round(4).to_dict(), lb_test=lb_test.round(4).to_dict(),
        test={k:{m:round(v,3) for m,v in metrics(te,p,tr).items()} for k,p in preds.items()})
    print(key, json.dumps(out[key],indent=1, default=str)); sys.stdout.flush()
json.dump(out,open('out/protocol.json','w'),indent=1,default=str)

In [ ]:
# Table 3 (training from 2020) and Table S2 (training from 2022): SARIMA vs naive benchmarks, raw counts
o = json.load(open('out/protocol.json'))
names = {'pre_raw':'SARIMA (2,1,2)(2,0,2)7','grid':'SARIMA, AIC-selected order','persistence':'Persistence (t−1)',
         'seasonal_naive':'Seasonal naive (t−7)','mean7':'Trailing 7-day mean','pre_log':'SARIMA, log1p + Duan (sensitivity)'}
def perf_table(window):
    rows = []
    for ph, lab in [('P1','Phase 1: test 2024'), ('P2','Phase 2: test 2023')]:
        P = pd.read_csv(f'out/preds_{window}_{ph}.csv', index_col=0, parse_dates=True)
        s0, e0, _ = split_years(window, ph); ytr = load()['Dengue_Cases'].astype(float)[s0:e0]
        for k, n in names.items():
            r = metrics(P.actual, P[k], ytr)          # computed directly from the forecasts (single rounding)
            rows.append({'Phase': lab, 'Model': n, **{m: round(v, 2) for m, v in r.items()}})
    return pd.DataFrame(rows)
table3 = perf_table('orig'); table3.to_csv('out/Table3_SARIMA_vs_benchmarks.csv', index=False); display(table3)
tableS2 = perf_table('restr'); tableS2.to_csv('out/TableS2_restricted_window.csv', index=False); display(tableS2)
sel = pd.DataFrame([{'Window/phase': k, 'Inner validation': ' to '.join(v['inner_val']), 'Inner MAE raw': round(v['inner_MAE']['raw'], 1),
        'Inner MAE log1p+Duan': round(v['inner_MAE']['log'], 1), 'Chosen': v['chosen_tf'], 'AIC-best order': v['best'],
        'Ljung–Box residual P (lag 7 / 14)': f"{v['lb_train']['lb_pvalue']['7']:.3f} / {v['lb_train']['lb_pvalue']['14']:.3f}"} for k, v in o.items()])
sel.to_csv('out/TableS2b_transformation_and_order_selection.csv', index=False); display(sel)

# Figure 5: rolling one-day-ahead SARIMA forecasts (raw counts)
fig, ax = plt.subplots(2, 1, figsize=(11, 7))
for a, (ph, lab) in zip(ax, [('P1', 'A. 2024 (typical year; trained 2020–2023)'), ('P2', 'B. 2023 (outbreak year; trained 2020–2022)')]):
    P = pd.read_csv(f'out/preds_orig_{ph}.csv', index_col=0, parse_dates=True)
    a.plot(P.index, P.actual, color='black', lw=1, label='Observed')
    a.plot(P.index, P.pre_raw, color='#c44e52', lw=1, ls='--', label='SARIMA one-day-ahead forecast')
    a.set_title(lab + f"   MAE = {metrics(P.actual, P.pre_raw)['MAE']:.1f}", loc='left', fontsize=11)
    a.set_ylabel('Daily admissions'); a.legend(frameon=False, loc='upper left', fontsize=9)
    a.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout(); plt.savefig('out/Fig5_rolling_SARIMA.png', dpi=600); plt.savefig('out/Fig5_rolling_SARIMA.svg'); plt.show()

## 5. Machine-learning comparison (Tables 4–5, Table S5, Figure S3)
(a) The submitted models: two features (Cases_7Day_Mean, Cases_Velocity_7Day), original hyperparameters, and the gated hybrid. Note that Cases_Velocity_7Day is a **difference** of 7-day means, not a percentage change; the Methods text and equation 6 must match this.
(b) The fair benchmark: all models on a common feature set (case lags 1–7, 14, 21 and 28 days; 7- and 14-day means; velocity; annual Fourier terms; day of week), tuned by 4-fold walk-forward cross-validation inside the training data, with a level target and a deviation-from-7-day-mean target. About 20–40 minutes.

In [ ]:
"""R1 #4: hybrid gating parameters + fair ML comparison (same information set, nested tuning in training only)."""
import json, sys, numpy as np, pandas as pd, warnings; warnings.filterwarnings('ignore')
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
SEED=42
df=load(); y=df.Dengue_Cases.astype(float)
F=pd.DataFrame(index=y.index)
F['Cases_7Day_Mean']=y.shift(1).rolling(7).mean()
F['Cases_Velocity_7Day']=F.Cases_7Day_Mean-y.shift(8).rolling(7).mean()
for L in list(range(1,8))+[14,21,28]: F[f'lag{L}']=y.shift(L)
F['Cases_14Day_Mean']=y.shift(1).rolling(14).mean()
t=(y.index-pd.Timestamp('2020-01-01')).days.values
for k in (1,2): F[f'sin{k}']=np.sin(2*np.pi*k*t/365.25); F[f'cos{k}']=np.cos(2*np.pi*k*t/365.25)
for d in range(6): F[f'dow{d}']=(y.index.dayofweek==d).astype(float)
LEAN=['Cases_7Day_Mean','Cases_Velocity_7Day']; RICH=list(F.columns)
F=F.dropna(); Y=y.loc[F.index]
def split(w,ph):
    s='2020' if w=='orig' else '2022'
    tr=F.loc[s:'2023'].index if ph=='P1' else F.loc[s:'2022'].index
    te=F.loc['2024'].index if ph=='P1' else F.loc['2023'].index
    return tr,te
ORIG={  # hyperparameters exactly as in the submitted notebook
 'Linear Regression':make_pipeline(StandardScaler(),LinearRegression()),
 'Ridge':make_pipeline(StandardScaler(),Ridge(alpha=1.0)),
 'SVR':make_pipeline(StandardScaler(),SVR(kernel='rbf',C=100,epsilon=5)),
 'MLP':make_pipeline(StandardScaler(),MLPRegressor(hidden_layer_sizes=(64,32),max_iter=2000,early_stopping=True,random_state=SEED)),
 'Random Forest':RandomForestRegressor(n_estimators=300,max_depth=6,min_samples_leaf=5,random_state=SEED,n_jobs=2),
 'XGBoost':XGBRegressor(n_estimators=120,max_depth=3,learning_rate=0.04,subsample=0.8,colsample_bytree=0.8,min_child_weight=5,random_state=SEED),
 'LightGBM':LGBMRegressor(n_estimators=200,max_depth=4,learning_rate=0.05,random_state=SEED,verbose=-1),
 'CatBoost':CatBoostRegressor(iterations=300,depth=4,learning_rate=0.05,random_seed=SEED,verbose=0)}
GRID={
 'Linear Regression':(make_pipeline(StandardScaler(),LinearRegression()),{}),
 'Ridge':(make_pipeline(StandardScaler(),Ridge()),{'ridge__alpha':[0.1,1,10,100]}),
 'SVR':(make_pipeline(StandardScaler(),SVR(kernel='rbf')),{'svr__C':[10,100,1000],'svr__epsilon':[1,5,10],'svr__gamma':['scale',0.01]}),
 'MLP':(make_pipeline(StandardScaler(),MLPRegressor(max_iter=2000,early_stopping=True,random_state=SEED)),
        {'mlpregressor__hidden_layer_sizes':[(32,),(64,32)],'mlpregressor__alpha':[1e-4,1e-2]}),
 'Random Forest':(RandomForestRegressor(n_estimators=300,random_state=SEED,n_jobs=2),{'max_depth':[4,8,None],'min_samples_leaf':[1,5,10]}),
 'XGBoost':(XGBRegressor(subsample=0.8,colsample_bytree=0.8,random_state=SEED),{'n_estimators':[200,500],'max_depth':[3,5],'learning_rate':[0.03,0.1]}),
 'LightGBM':(LGBMRegressor(random_state=SEED,verbose=-1),{'n_estimators':[200,500],'max_depth':[3,5],'learning_rate':[0.03,0.1]}),
 'CatBoost':(CatBoostRegressor(iterations=500,random_seed=SEED,verbose=0),{'depth':[4,6],'learning_rate':[0.03,0.1]})}
sig=lambda x:1/(1+np.exp(-x))
out={}; allpred={}
for w in ['orig','restr']:
  for ph in ['P1','P2']:
    key=f'{w}_{ph}'; tr,te=split(w,ph); r={}; P=pd.DataFrame(index=te,data={'actual':Y[te]})
    # (a) reproduction: lean features, original hyperparameters
    for n,m in ORIG.items():
        p=np.clip(m.fit(F.loc[tr,LEAN],Y[tr]).predict(F.loc[te,LEAN]),0,None); P[f'lean_{n}']=p
        r[f'lean_orig|{n}']=metrics(Y[te],p,Y[tr])
    # hybrid exactly as submitted
    c=np.percentile(F.loc[tr,'Cases_7Day_Mean'],95); lo=np.percentile(F.loc[tr,'Cases_7Day_Mean'],80)
    hi=np.percentile(F.loc[tr,'Cases_7Day_Mean'],99.9); s=(hi-lo)/6
    wlr=sig((F.loc[te,'Cases_7Day_Mean'].values-c)/s)
    P['hybrid']=(1-wlr)*P['lean_SVR']+wlr*P['lean_Linear Regression']
    r['lean_orig|Hybrid SVR+LR']=metrics(Y[te],P['hybrid'],Y[tr])
    gate=dict(c=round(c,2),P80=round(lo,2),P99_9=round(hi,2),s=round(s,2),
              train_max_7daymean=round(F.loc[tr,'Cases_7Day_Mean'].max(),1),
              test_max_7daymean=round(F.loc[te,'Cases_7Day_Mean'].max(),1),
              share_test_days_LR_weight_gt_0_5=round(float((wlr>0.5).mean()),3),
              share_test_days_beyond_train_max=round(float((F.loc[te,'Cases_7Day_Mean']>F.loc[tr,'Cases_7Day_Mean'].max()).mean()),3))
    # (b) fair benchmark: rich features, nested time-series CV tuning on training only; level and delta targets
    best={}
    for tgt in ['level','delta']:
      yt=Y[tr]-(F.loc[tr,'Cases_7Day_Mean'] if tgt=='delta' else 0)
      for n,(m,g) in GRID.items():
        gs=GridSearchCV(m,g,cv=TimeSeriesSplit(4),scoring='neg_mean_absolute_error',n_jobs=1).fit(F.loc[tr,RICH],yt)
        p=gs.predict(F.loc[te,RICH])+(F.loc[te,'Cases_7Day_Mean'].values if tgt=='delta' else 0)
        p=np.clip(p,0,None); P[f'rich_{tgt}_{n}']=p
        r[f'rich_{tgt}|{n}']=metrics(Y[te],p,Y[tr]); best[f'{tgt}|{n}']=gs.best_params_
      print(key,tgt,'done'); sys.stdout.flush()
    P.to_csv(f'out/mlpreds_{key}.csv')
    out[key]=dict(gate=gate,best_params=best,metrics={k:{m:round(v,3) for m,v in d.items()} for k,d in r.items()})
    print(key,json.dumps(out[key]['gate'])); 
    print(pd.DataFrame(out[key]['metrics']).T[['MAE','RMSE','sMAPE','R2']].to_string()); sys.stdout.flush()
json.dump(out,open('out/ml.json','w'),indent=1,default=str)

In [ ]:
# Tables 4–5: ML comparison (training from 2020); Table S5: gate values and tuned hyperparameters
mj = json.load(open('out/ml.json'))
def ml_table(key):
    P0 = pd.read_csv(f'out/preds_{key}.csv', index_col=0); s0, e0, _ = split_years(*key.split('_'))
    rows = [{'Model': 'SARIMA (2,1,2)(2,0,2)7', 'Feature set': 'Own lag structure',
             **{m: round(v, 2) for m, v in metrics(P0.actual, P0.pre_raw, load()['Dengue_Cases'].astype(float)[s0:e0]).items()}}]
    for k, v in mj[key]['metrics'].items():
        fs, model = k.split('|')
        rows.append({'Model': model, 'Feature set': {'lean_orig': '2 features, original settings', 'rich_level': 'Common set, level target',
                     'rich_delta': 'Common set, deviation target'}[fs], **{m: round(x, 2) for m, x in v.items()}})
    return pd.DataFrame(rows).sort_values('MAE')
table4 = ml_table('orig_P1'); table4.to_csv('out/Table4_models_2024.csv', index=False); display(table4.head(12))
table5 = ml_table('orig_P2'); table5.to_csv('out/Table5_models_2023.csv', index=False); display(table5.head(12))
gate = pd.DataFrame({k: v['gate'] for k, v in mj.items()}).T; gate.to_csv('out/TableS5a_hybrid_gate.csv'); display(gate)
params = pd.DataFrame([{'Window/phase': k, 'Target|model': m, 'Selected': json.dumps(p)} for k, v in mj.items() for m, p in v['best_params'].items()])
params.to_csv('out/TableS5b_tuned_hyperparameters.csv', index=False)

# Figure S3: hybrid model forecasts (as submitted)
fig, ax = plt.subplots(2, 1, figsize=(11, 7))
for a, (ph, lab) in zip(ax, [('P1', 'A. 2024'), ('P2', 'B. 2023')]):
    P = pd.read_csv(f'out/mlpreds_orig_{ph}.csv', index_col=0, parse_dates=True)
    a.plot(P.index, P.actual, color='black', lw=1, label='Observed'); a.plot(P.index, P.hybrid, color='#d35400', lw=1, ls='--', label='Hybrid SVR + linear regression')
    a.set_title(lab, loc='left'); a.set_ylabel('Daily admissions'); a.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.savefig('out/FigS3_hybrid.png', dpi=600); plt.show()

## 6. LSTM under the common protocol (Table S9)
Architecture as submitted (60-day window → LSTM → 16-unit ReLU → 1), Adam (learning rate 10⁻³), MSE loss, batch size 32, seed 42. Hidden size (16, 32 or 64) and epochs (early stopping, patience 10) chosen on the last 20% of the training data. Slow on CPU: about 30–60 minutes.

In [ ]:
"""R3 #6 / R4 #5: LSTM re-run under the common protocol. Architecture as submitted (LSTM 32 -> Dense 16 ReLU -> 1, 60-day window),
hidden size tuned on an inner-validation block of the training data, early stopping on that block; seed 42."""
import sys, json, numpy as np, pandas as pd, torch, torch.nn as nn
torch.set_num_threads(2)
y=load().Dengue_Cases.astype(float); W=60
class Net(nn.Module):
    def __init__(s,h): super().__init__(); s.l=nn.LSTM(1,h,batch_first=True); s.d=nn.Sequential(nn.Linear(h,16),nn.ReLU(),nn.Linear(16,1))
    def forward(s,x): o,_=s.l(x); return s.d(o[:,-1]).squeeze(-1)
def seqs(vals,idx_targets,scale,target):
    X=np.stack([vals[i-W:i] for i in idx_targets])/scale
    m7=np.array([vals[i-7:i].mean() for i in idx_targets])
    Yt=(vals[idx_targets]-(m7 if target=='delta' else 0))/scale
    return torch.tensor(X[...,None],dtype=torch.float32),torch.tensor(Yt,dtype=torch.float32),m7
def train(Xtr,Ytr,h,epochs,Xva=None,Yva=None,seed=42):
    torch.manual_seed(seed); np.random.seed(seed); net=Net(h); opt=torch.optim.Adam(net.parameters(),1e-3); lf=nn.MSELoss()
    best=(1e9,0,None); n=len(Xtr)
    for ep in range(1,epochs+1):
        net.train(); perm=torch.randperm(n)
        for b in range(0,n,32):
            i=perm[b:b+32]; opt.zero_grad(); l=lf(net(Xtr[i]),Ytr[i]); l.backward(); opt.step()
        if Xva is not None:
            net.eval(); 
            with torch.no_grad(): v=lf(net(Xva),Yva).item()
            if v<best[0]: best=(v,ep,{k:t.clone() for k,t in net.state_dict().items()})
            elif ep-best[1]>=10: break
    return net,best
vals=y.values; pos={d:i for i,d in enumerate(y.index)}
rows=[]; preds={}
for w in ['orig','restr']:
  for ph in ['P1','P2']:
    s='2020' if w=='orig' else '2022'; end='2023' if ph=='P1' else '2022'; test='2024' if ph=='P1' else '2023'
    tr_idx=np.array([pos[d] for d in y[s:end].index if pos[d]>=W]); te_idx=np.array([pos[d] for d in y[test].index])
    scale=vals[tr_idx].max()
    for target in ['level','delta']:
        cut=int(len(tr_idx)*0.8); best_cfg=None
        Xa,Ya,_=seqs(vals,tr_idx[:cut],scale,target); Xb,Yb,_=seqs(vals,tr_idx[cut:],scale,target)
        for h in (16,32,64):
            _,b=train(Xa,Ya,h,150,Xb,Yb)
            if best_cfg is None or b[0]<best_cfg[0]: best_cfg=(b[0],h,b[1])
        _,h,ep=best_cfg
        Xt,Yt,_=seqs(vals,tr_idx,scale,target); net,_=train(Xt,Yt,h,max(ep,5))
        Xe,_,m7=seqs(vals,te_idx,scale,target); net.eval()
        with torch.no_grad(): p=net(Xe).numpy()*scale+(m7 if target=='delta' else 0)
        p=np.clip(p,0,None); preds[f'{w}_{ph}_{target}']=p
        m=metrics(vals[te_idx],p,vals[tr_idx]); rows.append(dict(window=w,phase=ph,target=target,hidden=h,epochs=ep,**{k:round(v,3) for k,v in m.items()}))
        print(rows[-1]); sys.stdout.flush()
pd.DataFrame(rows).to_csv('out/lstm.csv',index=False)
for k,p in preds.items(): pd.Series(p).to_csv(f'out/lstm_pred_{k}.csv',index=False)

## 7. Formal forecast comparison (Table S8)
Diebold–Mariano tests (HLN correction, HAC variance, two-sided), 95% moving-block bootstrap intervals for MAE differences (negative = SARIMA better), and metrics for the high-transmission (Jul–Nov) and low-transmission (Dec–Jun) periods. The "best tuned ML" comparator is chosen on test MAE, which favours ML.

In [ ]:
"""R1 #7: Diebold-Mariano (HLN-corrected, HAC) tests, block-bootstrap CIs, stratified metrics."""
import json, numpy as np, pandas as pd
from scipy import stats
y=load().Dengue_Cases.astype(float)
def dm(e1,e2,loss='abs',lag=7):
    d=(np.abs(e1)-np.abs(e2)) if loss=='abs' else (e1**2-e2**2); T=len(d); dbar=d.mean(); dc=d-dbar
    g=[np.sum(dc[k:]*dc[:T-k])/T for k in range(lag+1)]
    lrv=g[0]+2*sum((1-k/(lag+1))*g[k] for k in range(1,lag+1))
    s=dbar/np.sqrt(lrv/T); h=1; s*=np.sqrt((T+1-2*h+h*(h-1)/T)/T)   # HLN correction
    return s, 2*stats.t.sf(abs(s),T-1)
def boot_ci(e1,e2,B=2000,block=14,seed=1):
    rng=np.random.default_rng(seed); d=np.abs(e1)-np.abs(e2); T=len(d); nb=int(np.ceil(T/block)); out=[]
    for _ in range(B):
        st=rng.integers(0,T-block+1,nb); idx=np.concatenate([np.arange(s,s+block) for s in st])[:T]; out.append(d[idx].mean())
    return np.percentile(out,[2.5,97.5])
# journal-style P values come from the helper pfmt()
rows=[]; strat=[]
for key in ['orig_P1','orig_P2','restr_P1','restr_P2']:
    a=pd.read_csv(f'out/preds_{key}.csv',index_col=0,parse_dates=True); b=pd.read_csv(f'out/mlpreds_{key}.csv',index_col=0,parse_dates=True)
    P=a.join(b.drop(columns='actual'))
    rich=[c for c in P if c.startswith('rich_')]; best_rich=min(rich,key=lambda c:np.mean(np.abs(P.actual-P[c])))
    comps={'Persistence':'persistence','Seasonal naive (t-7)':'seasonal_naive','7-day mean':'mean7',
           'Linear Regression (2 features)':'lean_Linear Regression','Hybrid SVR+LR':'hybrid',
           f'Best tuned ML ({best_rich.replace("rich_","")})':best_rich}
    e0=(P.actual-P.pre_raw).values
    for name,c in comps.items():
        e1=(P.actual-np.clip(P[c],0,None)).values
        sa,pa=dm(e0,e1,'abs'); ss,ps=dm(e0,e1,'sq'); lo,hi=boot_ci(e0,e1)
        rows.append(dict(split=key,comparator=name,MAE_SARIMA=round(np.mean(np.abs(e0)),2),MAE_comp=round(np.mean(np.abs(e1)),2),
             MAE_diff=round(np.mean(np.abs(e0))-np.mean(np.abs(e1)),2),CI95=f'{lo:.2f} to {hi:.2f}',
             DM_abs=round(sa,2),P_abs=pfmt(pa),DM_sq=round(ss,2),P_sq=pfmt(ps)))
    trn=y['2020' if key.startswith('orig') else '2022':('2023' if key.endswith('P1') else '2022')]
    for per,mask in [('High transmission (Jul-Nov)',P.index.month.isin(range(7,12))),('Low transmission (Dec-Jun)',~P.index.month.isin(range(7,12)))]:
        for name,c in [('SARIMA','pre_raw'),('Persistence','persistence'),('Seasonal naive','seasonal_naive'),('Hybrid SVR+LR','hybrid'),(best_rich.replace('rich_',''),best_rich)]:
            m=metrics(P.actual[mask],P[c][mask],trn); strat.append(dict(split=key,period=per,n_days=int(mask.sum()),model=name,**{k:round(v,3) for k,v in m.items()}))
R=pd.DataFrame(rows); S=pd.DataFrame(strat)
print(R.to_string(index=False)); print(); print(S.to_string(index=False))
R.to_csv('out/dm_tests.csv',index=False); S.to_csv('out/stratified.csv',index=False)

In [ ]:
# Add the LSTM (deviation target) to the formal comparison; build Table S8 and Table S9
rows = []
yv = load()['Dengue_Cases'].astype(float)
for w in ['orig', 'restr']:
    for ph in ['P1', 'P2']:
        A = pd.read_csv(f'out/preds_{w}_{ph}.csv', index_col=0, parse_dates=True); e0 = (A.actual - A.pre_raw).values
        for t in ['level', 'delta']:
            L = pd.read_csv(f'out/lstm_pred_{w}_{ph}_{t}.csv').iloc[:, 0].values; e1 = A.actual.values - L
            s, p = dm_test(e0, e1); lo, hi = boot_ci(e0, e1)
            rows.append({'split': f'{w}_{ph}', 'comparator': f'LSTM ({t} target)', 'MAE_SARIMA': round(np.mean(abs(e0)), 2), 'MAE_comp': round(np.mean(abs(e1)), 2),
                         'MAE_diff': round(np.mean(abs(e0)) - np.mean(abs(e1)), 2), 'CI95': f'{lo:.2f} to {hi:.2f}', 'DM_abs': round(s, 2), 'P_abs': pfmt(p)})
S8 = pd.concat([pd.read_csv('out/dm_tests.csv'), pd.DataFrame(rows)], ignore_index=True)
S8.to_csv('out/TableS8_forecast_comparison_tests.csv', index=False); display(S8)
pd.read_csv('out/stratified.csv').to_csv('out/TableS8b_stratified_metrics.csv', index=False)
pd.read_csv('out/lstm.csv').to_csv('out/TableS9_LSTM.csv', index=False)

## 8. Weather variables: three separate questions (Tables S3, S4, S10)
**8a. Incremental predictive information (Granger):** weekly data, seasonality removed with an annual Fourier fit, lags of 4–12 weeks, all three variables.

In [ ]:
"""R1 #2: Granger tests for all three weather variables on deseasonalised WEEKLY data, lags 1-12 weeks."""
import numpy as np, pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
df=load()
w=pd.DataFrame({'cases':np.log1p(df.Dengue_Cases.resample('W-SUN').sum()),
                'Temperature':df.Temperature.resample('W-SUN').mean(),
                'Humidity':df.Humidity.resample('W-SUN').mean(),
                'Rainfall':np.log1p(df.Rainfall_mm.resample('W-SUN').sum())})
w=w.iloc[1:-1]  # drop partial weeks
def deseason(s):
    t=np.arange(len(s)); X=[np.ones_like(t,float)]
    for k in (1,2): X+= [np.sin(2*np.pi*k*t/52.18), np.cos(2*np.pi*k*t/52.18)]
    X=np.column_stack(X); b=np.linalg.lstsq(X,s.values,rcond=None)[0]; return s-X@b
rows=[]
for label,sub in [('2020-2024',w),('2022-2024',w['2022':])]:
    d=sub.apply(deseason)
    # difference if non-stationary by ADF
    for c in d: 
        if adfuller(d[c].dropna())[1]>0.05: d[c]=d[c].diff()
    d=d.dropna()
    for v in ['Temperature','Humidity','Rainfall']:
        r=grangercausalitytests(d[['cases',v]],maxlag=12)
        for L in [4,6,8,10,12]:
            F,p,dn,dd=r[L][0]['ssr_ftest']
            rows.append(dict(period=label,variable=v,lag_weeks=L,F=round(F,2),df=f'{int(dd)},{int(dn)}',p=p,n_weeks=len(d)))
g=pd.DataFrame(rows)
g['p_fmt']=g.p.map(lambda p:'<.001' if p<.001 else (f'{p:.3f}' if p<.01 else f'{p:.2f}'))
print(g.drop(columns='p').to_string(index=False)); g.to_csv('out/granger_weekly.csv',index=False)
# lagged correlations (deseasonalised) for context
d=w.apply(deseason)
print('\nCross-corr deseasonalised weekly cases vs weather lagged k weeks')
print(pd.DataFrame({v:[d.cases.corr(d[v].shift(k)) for k in range(0,13)] for v in ['Temperature','Humidity','Rainfall']}).round(2).T)

**8b. Out-of-sample value in SARIMA:** SARIMA vs SARIMAX with lagged 7-day weather means (lag of 28, 56 or 84 days chosen on inner validation).

In [ ]:
"""R1 #2: out-of-sample SARIMA vs SARIMAX-X (lagged weather). Lag chosen on inner validation only."""
import json, numpy as np, pandas as pd
from joblib import Parallel, delayed
df=load(); y=df.Dengue_Cases.astype(float); y.index.freq='D'
base={'Temperature':df.Temperature.rolling(7).mean(),'Humidity':df.Humidity.rolling(7).mean(),
      'Rainfall':np.log1p(df.Rainfall_mm.rolling(7).sum())}
def X(vars_,L):
    return pd.concat({f'{v}_l{L}':base[v].shift(L) for v in vars_},axis=1)
sets={'none':[], 'Temperature':['Temperature'],'Humidity':['Humidity'],'Rainfall':['Rainfall'],
      'All three':['Temperature','Humidity','Rainfall']}
LAGS=[28,56,84]
def split(window,phase):
    s='2020-04-01' if window=='orig' else '2022'   # orig starts after 84-day lag burn-in
    return (y[s:'2023'],y['2024']) if phase=='P1' else (y[s:'2022'],y['2023'])
def inner(tr):
    n=365 if len(tr)>=730 else 184; return tr.iloc[:-n],tr.iloc[-n:]
def run(tr,te,vars_,L):
    if not vars_:
        r=fit_sarima(tr); return metrics(te,onestep(r,te))['MAE'], metrics(te,onestep(r,te))
    ex=X(vars_,L); r=fit_sarima(tr,exog=ex.loc[tr.index])
    p=onestep(r,te,ex.loc[te.index]); return metrics(te,p)['MAE'], metrics(te,p)
jobs=[]
for w in ['orig','restr']:
  for ph in ['P1','P2']:
    tr,te=split(w,ph); itr,iva=inner(tr)
    for s,v in sets.items():
      for L in (LAGS if v else [0]):
        jobs.append((w,ph,s,L,'inner',itr,iva,v)); jobs.append((w,ph,s,L,'test',tr,te,v))
res=Parallel(n_jobs=2)(delayed(run)(a,b,v,L) for (w,ph,s,L,k,a,b,v) in jobs)
rows=[dict(window=w,phase=ph,set=s,lag=L,stage=k,**{m:round(x,3) for m,x in r[1].items()})
      for (w,ph,s,L,k,a,b,v),r in zip(jobs,res)]
d=pd.DataFrame(rows); d.to_csv('out/weather_oos.csv',index=False)
# lag selected on inner validation, then report test
sel=[]
for (w,ph,s),g in d.groupby(['window','phase','set']):
    gi=g[g.stage=='inner']; L=gi.loc[gi.MAE.idxmin(),'lag']
    t=g[(g.stage=='test')&(g.lag==L)].iloc[0]
    sel.append(dict(window=w,phase=ph,set=s,lag_selected=L,inner_MAE=gi.MAE.min(),test_MAE=t.MAE,test_RMSE=t.RMSE))
print(pd.DataFrame(sel).to_string(index=False)); pd.DataFrame(sel).to_csv('out/weather_oos_selected.csv',index=False)

**8c. Out-of-sample value in ML models:** six models re-tuned with and without nine lagged weather features. About 20–40 minutes.

In [ ]:
"""R3 #5: do weather variables add value inside ML models? Same tuning protocol as R1 (training-only walk-forward CV)."""
import json, sys, numpy as np, pandas as pd, warnings; warnings.filterwarnings('ignore')
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
SEED=42; df=load(); y=df.Dengue_Cases.astype(float)
F=pd.DataFrame(index=y.index)
F['Cases_7Day_Mean']=y.shift(1).rolling(7).mean()
F['Cases_Velocity_7Day']=F.Cases_7Day_Mean-y.shift(8).rolling(7).mean()
for L in list(range(1,8))+[14,21,28]: F[f'lag{L}']=y.shift(L)
F['Cases_14Day_Mean']=y.shift(1).rolling(14).mean()
t=(y.index-pd.Timestamp('2020-01-01')).days.values
for k in (1,2): F[f'sin{k}']=np.sin(2*np.pi*k*t/365.25); F[f'cos{k}']=np.cos(2*np.pi*k*t/365.25)
for d in range(6): F[f'dow{d}']=(y.index.dayofweek==d).astype(float)
RICH=list(F.columns)
wx={'Temp':df.Temperature.rolling(7).mean(),'Hum':df.Humidity.rolling(7).mean(),'Rain':np.log1p(df.Rainfall_mm.rolling(7).sum())}
for v,s in wx.items():
    for L in (28,56,84): F[f'{v}_l{L}']=s.shift(L)
WX=RICH+[c for c in F if c.endswith(('_l28','_l56','_l84'))]
F=F.dropna(); Y=y.loc[F.index]
def split(w,ph):
    s='2020' if w=='orig' else '2022'
    tr=F.loc[s:'2023'].index if ph=='P1' else F.loc[s:'2022'].index
    te=F.loc['2024'].index if ph=='P1' else F.loc['2023'].index
    return tr,te
GRID={
 'Ridge':(make_pipeline(StandardScaler(),Ridge()),{'ridge__alpha':[0.1,1,10,100]}),
 'SVR':(make_pipeline(StandardScaler(),SVR(kernel='rbf')),{'svr__C':[10,100,1000],'svr__epsilon':[1,5,10],'svr__gamma':['scale',0.01]}),
 'Random Forest':(RandomForestRegressor(n_estimators=300,random_state=SEED,n_jobs=2),{'max_depth':[4,8,None],'min_samples_leaf':[1,5,10]}),
 'XGBoost':(XGBRegressor(subsample=0.8,colsample_bytree=0.8,random_state=SEED),{'n_estimators':[200,500],'max_depth':[3,5],'learning_rate':[0.03,0.1]}),
 'LightGBM':(LGBMRegressor(random_state=SEED,verbose=-1),{'n_estimators':[200,500],'max_depth':[3,5],'learning_rate':[0.03,0.1]}),
 'CatBoost':(CatBoostRegressor(iterations=500,random_seed=SEED,verbose=0),{'depth':[4,6],'learning_rate':[0.03,0.1]})}
rows=[]
for w in ['orig','restr']:
  for ph in ['P1','P2']:
    tr,te=split(w,ph); P=pd.DataFrame(index=te,data={'actual':Y[te]})
    yt=Y[tr]-F.loc[tr,'Cases_7Day_Mean']
    for fs,cols in [('cases only',RICH),('cases + weather',WX)]:
      for n,(m,g) in GRID.items():
        gs=GridSearchCV(m,g,cv=TimeSeriesSplit(4),scoring='neg_mean_absolute_error').fit(F.loc[tr,cols],yt)
        p=np.clip(gs.predict(F.loc[te,cols])+F.loc[te,'Cases_7Day_Mean'].values,0,None); P[f'{fs}|{n}']=p
        rows.append(dict(window=w,phase=ph,features=fs,model=n,cv_MAE=round(-gs.best_score_,2),**{k:round(v,3) for k,v in metrics(Y[te],p,Y[tr]).items()}))
      print(w,ph,fs,'done'); sys.stdout.flush()
    P.to_csv(f'out/mlwx_{w}_{ph}.csv')
R=pd.DataFrame(rows); R.to_csv('out/ml_weather.csv',index=False)
piv=R.pivot_table(index=['window','phase','model'],columns='features',values='MAE')
piv['change_%']=(piv['cases + weather']/piv['cases only']-1)*100
print(piv.round(2).to_string())

In [ ]:
pd.read_csv('out/granger_weekly.csv').to_csv('out/TableS3_granger.csv', index=False)
pd.read_csv('out/weather_oos_selected.csv').to_csv('out/TableS4_SARIMAX_weather.csv', index=False)
if not os.path.exists('out/ml_weather.csv'):
    raise RuntimeError("out/ml_weather.csv not found: run the previous cell (8c, ML models with/without weather) "
                       "and wait until it prints its results table before running this cell.")
R = pd.read_csv('out/ml_weather.csv')
piv = R.pivot_table(index=['window', 'phase', 'model'], columns='features', values='MAE')
piv['change_%'] = (piv['cases + weather']/piv['cases only'] - 1)*100
piv.round(2).to_csv('out/TableS10_ML_with_weather.csv'); display(piv.round(2))
print(f"MAE change from adding weather: {piv['change_%'].min():.1f}% to {piv['change_%'].max():.1f}%")

## 9. Forecast horizons of 7, 14 and 28 days (Table S11, Figure S2)
For every test day, a forecast from the origin h days earlier using only data available at that origin: iterated SARIMA, persistence, 7-day mean, day-of-year climatology, and a direct XGBoost model per horizon.

In [ ]:
"""R4 #2: multi-step forecasts at 7, 14, 28 days ahead from rolling origins (fixed parameters), vs benchmarks and direct ML."""
import sys, json, numpy as np, pandas as pd, warnings; warnings.filterwarnings('ignore')
from scipy import stats
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
y=load().Dengue_Cases.astype(float); y.index.freq='D'
H=[7,14,28]
def dm(e1,e2,h):
    d=np.abs(e1)-np.abs(e2); T=len(d); lag=max(h-1,7); dc=d-d.mean()
    g=[np.sum(dc[k:]*dc[:T-k])/T for k in range(lag+1)]; lrv=g[0]+2*sum((1-k/(lag+1))*g[k] for k in range(1,lag+1))
    s=d.mean()/np.sqrt(lrv/T)*np.sqrt((T+1-2*h+h*(h-1)/T)/T); return s,2*stats.t.sf(abs(s),T-1)
# direct ML features at origin t (all known at t)
def feats(h):
    F=pd.DataFrame(index=y.index)
    for L in list(range(0,7))+[13,20,27]: F[f'lag{L}']=y.shift(L)
    F['m7']=y.rolling(7).mean(); F['m14']=y.rolling(14).mean(); F['vel']=F.m7-y.shift(7).rolling(7).mean()
    tt=(y.index+pd.Timedelta(days=h)-pd.Timestamp('2020-01-01')).days.values
    for k in (1,2): F[f's{k}']=np.sin(2*np.pi*k*tt/365.25); F[f'c{k}']=np.cos(2*np.pi*k*tt/365.25)
    F['target']=y.shift(-h)-F.m7      # deviation target, indexed by origin
    return F
rows=[]
for w,s in [('orig','2020'),('restr','2022')]:
  for ph in ['P1','P2']:
    end='2023' if ph=='P1' else '2022'; test='2024' if ph=='P1' else '2023'
    tr=y[s:end]; te=y[test]; res=fit_sarima(tr); full=res.append(te,refit=False)
    n0=len(tr)                              # position of first test day in full
    out={h:{} for h in H}
    # SARIMA: for each target day in test, forecast from origin = target-h (origins may lie in training tail)
    for h in H:
        preds=[]
        for j in range(len(te)):
            o=n0+j-h                         # origin index (last observed)
            p=full.get_prediction(start=o+1,end=o+h,dynamic=0).predicted_mean.iloc[-1]
            preds.append(max(p,0))
        out[h]['SARIMA']=np.array(preds)
        tgt=te.index; org=tgt-pd.Timedelta(days=h)
        out[h]['Persistence']=y.reindex(org).values
        # seasonal naive: same weekday, latest week known at origin
        k=int(np.ceil(h/7))*7; out[h]['Seasonal naive']=y.reindex(tgt-pd.Timedelta(days=k)).values
        out[h]['7-day mean']=y.rolling(7).mean().reindex(org).values
        # climatology: mean of same calendar day-of-year +/-3 days in training years
        clim=tr.groupby(tr.index.dayofyear).mean().reindex(range(1,367)).interpolate()
        cl=pd.concat([clim]*3).rolling(7,center=True).mean().iloc[366:732].values
        out[h]['Climatology']=np.array([cl[d.dayofyear-1] for d in tgt])
        # direct XGBoost (tuned on training origins only)
        F=feats(h); Ftr=F.loc[s:end].dropna(); Ftr=Ftr[Ftr.index+pd.Timedelta(days=h)<=pd.Timestamp(end+'-12-31')]
        X=[c for c in F if c!='target']
        gs=GridSearchCV(XGBRegressor(subsample=0.8,colsample_bytree=0.8,random_state=42),
             {'n_estimators':[200,500],'max_depth':[3,5],'learning_rate':[0.03,0.1]},cv=TimeSeriesSplit(4),scoring='neg_mean_absolute_error').fit(Ftr[X],Ftr.target)
        Fo=F.reindex(org)[X]; out[h]['XGBoost (direct)']=np.clip(gs.predict(Fo)+Fo.m7.values,0,None)
        yt=te.values; e0=yt-out[h]['SARIMA']; hi=te.index.month.isin(range(7,12))
        for m,p in out[h].items():
            e=yt-p; mt=metrics(yt,p,tr.values); r=dict(window=w,phase=ph,h=h,model=m,MAE=round(mt['MAE'],2),RMSE=round(mt['RMSE'],2),MASE=round(mt['MASE'],3),
                 MAE_JulNov=round(np.mean(np.abs(e[hi])),2),MAE_DecJun=round(np.mean(np.abs(e[~hi])),2))
            if m!='SARIMA': sd,pv=dm(e0,e,h); r.update(DM=round(sd,2),P=pv)
            rows.append(r)
        print(w,ph,h,'done',{m:round(np.mean(np.abs(yt-p)),1) for m,p in out[h].items()}); sys.stdout.flush()
R=pd.DataFrame(rows); R.to_csv('out/horizons.csv',index=False); print(R.to_string(index=False))

In [ ]:
o = json.load(open('out/protocol.json')); mj = json.load(open('out/ml.json'))   # reload: names reused in earlier cells
R = pd.read_csv('out/horizons.csv'); R.to_csv('out/TableS11_horizons.csv', index=False)
h1 = {ph: {'SARIMA': o[f'orig_{ph}']['test']['pre_raw']['MAE'], 'Persistence': o[f'orig_{ph}']['test']['persistence']['MAE'],
           '7-day mean': o[f'orig_{ph}']['test']['mean7']['MAE'],
           'XGBoost (direct)': mj[f'orig_{ph}']['metrics']['rich_delta|XGBoost']['MAE']} for ph in ['P1', 'P2']}
st = {'SARIMA': ('#1f3b57','o','-'), 'Persistence': ('#e08e45','s','--'), '7-day mean': ('#8c8c8c','^',':'), 'XGBoost (direct)': ('#2e8b57','D','-.')}
Ro = R[R.window == 'orig']
fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
for a, (ph, lab) in zip(ax, [('P1', 'A. 2024 (typical year)'), ('P2', 'B. 2023 (outbreak year)')]):
    for m, (c, mk, ls) in st.items():
        v = [h1[ph][m]] + [Ro[(Ro.phase == ph) & (Ro.model == m) & (Ro.h == h)].MAE.iloc[0] for h in (7, 14, 28)]
        a.plot([1, 7, 14, 28], v, color=c, marker=mk, ls=ls, label=m)
    a.set_xticks([1, 7, 14, 28]); a.set_xlabel('Forecast horizon (days ahead)'); a.set_ylabel('MAE (admissions per day)'); a.set_title(lab, loc='left')
ax[0].legend(frameon=False, fontsize=9); plt.tight_layout(); plt.savefig('out/FigS2_horizons.png', dpi=600); plt.show()

**9b. Does weather help at longer horizons? (Table S12)**
The 7-, 14- and 28-day forecasts repeated with lagged temperature, humidity and log-transformed rainfall added to SARIMA (as SARIMAX) and to the direct XGBoost model. Positive change = weather increases error. About 15–25 minutes.

In [ ]:
# Longer horizons with vs without lagged weather (Table S12).
# SARIMA vs SARIMAX (all three weather variables, lag chosen on inner validation in Section 8b; lag >= 28 days, so
# the weather values needed for every forecast are already known at the forecast origin), and direct XGBoost with vs
# without weather known at the origin (lags 0, 28 and 56 days). Same origins, targets and tuning as Section 9.
y = load().Dengue_Cases.astype(float); y.index.freq = 'D'; df = load()
H = [7, 14, 28]
wbase = {'Temperature': df.Temperature.rolling(7).mean(), 'Humidity': df.Humidity.rolling(7).mean(),
         'Rainfall': np.log1p(df.Rainfall_mm.rolling(7).sum())}
sel = pd.read_csv('out/weather_oos_selected.csv'); sel = sel[sel.set == 'All three'].set_index(['window', 'phase']).lag_selected
def feats_w(h):
    F = pd.DataFrame(index=y.index)
    for L in list(range(0, 7)) + [13, 20, 27]: F[f'lag{L}'] = y.shift(L)
    F['m7'] = y.rolling(7).mean(); F['m14'] = y.rolling(14).mean(); F['vel'] = F.m7 - y.shift(7).rolling(7).mean()
    tt = (y.index + pd.Timedelta(days=h) - pd.Timestamp('2020-01-01')).days.values
    for k in (1, 2): F[f's{k}'] = np.sin(2*np.pi*k*tt/365.25); F[f'c{k}'] = np.cos(2*np.pi*k*tt/365.25)
    for v, s in wbase.items():
        for L in (0, 28, 56): F[f'{v}_o{L}'] = s.shift(L)
    F['target'] = y.shift(-h) - F.m7
    return F
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
rows = []
for w, s in [('orig', '2020-04-01'), ('restr', '2022')]:          # orig starts after the 84-day weather-lag burn-in
    for ph in ['P1', 'P2']:
        end = '2023' if ph == 'P1' else '2022'; test = '2024' if ph == 'P1' else '2023'
        L = int(sel[(w, ph)]); EX = pd.concat({f'{v}_l{L}': b.shift(L) for v, b in wbase.items()}, axis=1)
        tr = y[s:end]; te = y[test]; n0 = len(tr)
        f0 = fit_sarima(tr).append(te, refit=False)
        f1 = fit_sarima(tr, exog=EX.loc[tr.index]).append(te, exog=EX.loc[te.index], refit=False)
        for h in H:
            P = {}
            for nm, full in [('SARIMA', f0), ('SARIMAX (weather)', f1)]:
                P[nm] = np.array([max(full.get_prediction(start=n0+j-h+1, end=n0+j, dynamic=0).predicted_mean.iloc[-1], 0)
                                  for j in range(len(te))])
            org = te.index - pd.Timedelta(days=h); F = feats_w(h)
            for nm, wx in [('XGBoost (direct)', False), ('XGBoost (direct, weather)', True)]:
                X = [c for c in F if c != 'target' and (wx or '_o' not in c)]
                Ftr = F.loc[s:end].dropna(); Ftr = Ftr[Ftr.index + pd.Timedelta(days=h) <= pd.Timestamp(end + '-12-31')]
                gs = GridSearchCV(XGBRegressor(subsample=0.8, colsample_bytree=0.8, random_state=SEED),
                                  {'n_estimators': [200, 500], 'max_depth': [3, 5], 'learning_rate': [0.03, 0.1]},
                                  cv=TimeSeriesSplit(4), scoring='neg_mean_absolute_error').fit(Ftr[X], Ftr.target)
                Fo = F.reindex(org)[X]; P[nm] = np.clip(gs.predict(Fo) + Fo.m7.values, 0, None)
            yt = te.values; hi = te.index.month.isin(range(7, 12))
            for nm, p in P.items():
                e = yt - p; rows.append(dict(window=w, phase=ph, h=h, model=nm, MAE=round(np.mean(np.abs(e)), 2),
                                             MAE_JulNov=round(np.mean(np.abs(e[hi])), 2)))
            for a, b in [('SARIMA', 'SARIMAX (weather)'), ('XGBoost (direct)', 'XGBoost (direct, weather)')]:
                ea, eb = yt - P[a], yt - P[b]; sd, pv = dm_test(eb, ea, h)    # positive = weather worse
                ma, mb = np.mean(np.abs(ea)), np.mean(np.abs(eb))
                rows.append(dict(window=w, phase=ph, h=h, model=f'{b} vs {a}', MAE=round(mb-ma, 2),
                                 change_pct=round((mb/ma-1)*100, 1), DM=round(sd, 2), P=pv))
            print(w, ph, h, {k: round(np.mean(np.abs(yt-v)), 1) for k, v in P.items()})
pd.DataFrame(rows).to_csv('out/horizons_weather.csv', index=False)

In [ ]:
R = pd.read_csv('out/horizons_weather.csv'); rows = []
for (w, ph, h), g in R.groupby(['window', 'phase', 'h'], sort=False):
    m = g.set_index('model'); s = m.loc['SARIMAX (weather) vs SARIMA']; x = m.loc['XGBoost (direct, weather) vs XGBoost (direct)']
    rows.append({'Training from': '2020' if w == 'orig' else '2022', 'Test year': '2024' if ph == 'P1' else '2023', 'Horizon (days)': h,
                 'SARIMA': m.loc['SARIMA', 'MAE'], 'SARIMA + weather': m.loc['SARIMAX (weather)', 'MAE'],
                 'SARIMA change (%)': s.change_pct, 'SARIMA P': pfmt(s.P),
                 'XGBoost': m.loc['XGBoost (direct)', 'MAE'], 'XGBoost + weather': m.loc['XGBoost (direct, weather)', 'MAE'],
                 'XGBoost change (%)': x.change_pct, 'XGBoost P': pfmt(x.P)})
S12 = pd.DataFrame(rows); S12.to_csv('out/TableS12_horizons_weather.csv', index=False); display(S12)
ch = pd.concat([S12['SARIMA change (%)'], S12['XGBoost change (%)']])
print(f'Weather lowered MAE in {(ch < 0).sum()} of {len(ch)} comparisons and raised it in {(ch > 0).sum()}.')
cmp_ = R[R.model.str.contains(' vs ')]
print(f'Significantly worse with weather (P<.05): {((cmp_.change_pct > 0) & (cmp_.P < .05)).sum()}; significantly better: {((cmp_.change_pct < 0) & (cmp_.P < .05)).sum()}.')

## 10. 2027 long-range scenario (Tables S6–S7, Figure 6)
Fourier order chosen by AIC; hindcasts from origins at the end of 2021, 2022 and 2023 compared with a climatological benchmark; 2027 projection with 80% and 95% prediction intervals from 1,000 simulated paths. The submitted d = 1 model gives unbounded intervals, so the stationary d = 0 version is the primary projection.

In [ ]:
"""R1 #5: Fourier-augmented SARIMA long-horizon projection: K selection, simulation PIs, pseudo-out-of-sample hindcasts."""
import json, sys, numpy as np, pandas as pd, warnings; warnings.filterwarnings('ignore')
from statsmodels.tsa.statespace.sarimax import SARIMAX
df=load(); y=df.Dengue_Cases.astype(float); y.index.freq='D'
T0=pd.Timestamp('2020-01-01')
def four(idx,K):
    t=(idx-T0).days.values; d={}
    for k in range(1,K+1): d[f's{k}']=np.sin(2*np.pi*k*t/365.25); d[f'c{k}']=np.cos(2*np.pi*k*t/365.25)
    return pd.DataFrame(d,index=idx)
SPECS={'d1 (as submitted)':((2,1,2),(2,0,2,7)), 'd0 (stationary)':((2,0,2),(1,0,1,7))}
def fit(ytr,K,spec):
    o,so=SPECS[spec]
    return SARIMAX(ytr,exog=four(ytr.index,K),order=o,seasonal_order=so,trend='c' if spec.startswith('d0') else None,
                   enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=300)
def monthly_paths(res,ytr,end,nsim=500,seed=7):
    """Simulate future paths with explicitly drawn standard-normal shocks, so results are reproducible
    across statsmodels versions (the simulate() random-number API differs between versions)."""
    rng=np.random.default_rng(seed)
    idx=pd.date_range(ytr.index[-1]+pd.Timedelta(days=1),end,freq='D'); ex=four(idx,res.model.k_exog//2)
    mean=res.get_forecast(len(idx),exog=ex).predicted_mean.clip(lower=0); mean.index=idx
    ke,kp=res.model.k_endog,res.model.ssm.k_posdef; paths=[]
    for _ in range(nsim):
        sim=res.simulate(len(idx),anchor='end',exog=ex,
                         measurement_shocks=rng.standard_normal((len(idx),ke)),state_shocks=rng.standard_normal((len(idx),kp)),
                         pretransformed_measurement_shocks=False,pretransformed_state_shocks=False)
        paths.append(np.clip(np.asarray(sim).ravel(),0,None))
    return mean, pd.DataFrame(np.column_stack(paths),index=idx)
out={}
# 1) K selection by AIC, both data windows, both specs
for w,s in [('orig','2020'),('restr','2022')]:
  for spec in SPECS:
    a={K:round(fit(y[s:],K,spec).aic,1) for K in [1,2,3,4]}; out[f'aic|{w}|{spec}']=a; print(w,spec,a); sys.stdout.flush()
# 2) hindcasts: fit to data ending at origin, forecast later calendar years; compare with climatology benchmark
hind=[]
for origin_end,targets in [('2021-12-31',[2023,2024]),('2022-12-31',[2023,2024]),('2023-12-31',[2024])]:
  ytr=y[:origin_end]
  clim=ytr.groupby([ytr.index.year,ytr.index.month]).sum().groupby(level=1).mean()   # mean monthly profile of training yrs
  for spec in SPECS:
    res=fit(ytr,2,spec); mean,sims=monthly_paths(res,ytr,'2024-12-31',nsim=300)
    for T in targets:
      obs=y[str(T)].groupby(y[str(T)].index.month).sum(); fm=mean[str(T)].groupby(mean[str(T)].index.month).sum()
      tot=sims.loc[str(T)].sum(); lo,hi=np.percentile(tot,[2.5,97.5])
      hind.append(dict(origin=origin_end[:4],target=T,horizon_yrs=T-int(origin_end[:4]),spec=spec,
        obs_total=int(obs.sum()),fc_total=int(fm.sum()),PI95=f'{int(lo)}-{int(hi)}',covered=bool(lo<=obs.sum()<=hi),
        obs_peak=int(obs.idxmax()),fc_peak=int(fm.idxmax()),monthly_MAE=round(np.mean(np.abs(obs-fm)),0),
        clim_total=int(clim.sum()),clim_peak=int(clim.idxmax()),clim_monthly_MAE=round(np.mean(np.abs(obs-clim)),0)))
  print(origin_end,'done'); sys.stdout.flush()
H=pd.DataFrame(hind); print(H.to_string(index=False)); H.to_csv('out/hindcast.csv',index=False)
# 3) 2027 projection with simulation-based 95% PIs
proj={}
for w,s in [('orig','2020'),('restr','2022')]:
  for spec in SPECS:
    ytr=y[s:]; res=fit(ytr,2,spec); mean,sims=monthly_paths(res,ytr,'2027-12-31',nsim=1000)
    m=mean['2027'].groupby(mean['2027'].index.month).sum()
    ms=sims.loc['2027'].groupby(sims.loc['2027'].index.month).sum()
    tab=pd.DataFrame({'mean':m.round(0),'lo95':ms.quantile(0.025,axis=1).round(0),'hi95':ms.quantile(0.975,axis=1).round(0),
                      'lo80':ms.quantile(0.10,axis=1).round(0),'hi80':ms.quantile(0.90,axis=1).round(0)})
    tot=sims.loc['2027'].sum()
    proj[f'{w}|{spec}']=dict(monthly=tab.to_dict(),total=int(m.sum()),total_PI95=[int(np.percentile(tot,2.5)),int(np.percentile(tot,97.5))],
        total_PI80=[int(np.percentile(tot,10)),int(np.percentile(tot,90))],peak=int(m.idxmax()))
    print(w,spec,'total',int(m.sum()),proj[f'{w}|{spec}']['total_PI95'],'peak',int(m.idxmax())); print(tab.to_string()); sys.stdout.flush()
    tab.to_csv(f'out/proj2027_{w}_{spec.split()[0]}.csv')
out['proj']=proj; json.dump(out,open('out/proj2027.json','w'),indent=1,default=str)

In [ ]:
pd.read_csv('out/hindcast.csv').to_csv('out/TableS6_hindcasts.csv', index=False)
pd.read_csv('out/proj2027_orig_d0.csv').to_csv('out/TableS7_2027_monthly.csv', index=False)
pj = json.load(open('out/proj2027.json'))['proj']['orig|d0 (stationary)']
print(f"2027 scenario total: {pj['total']:,} (80% PI {pj['total_PI80'][0]:,}–{pj['total_PI80'][1]:,}; 95% PI {pj['total_PI95'][0]:,}–{pj['total_PI95'][1]:,}); peak month {pj['peak']}")

In [ ]:
y=load().Dengue_Cases

fig,ax=plt.subplots(figsize=(10,4.8))
d0=pd.read_csv('out/proj2027_orig_d0.csv',index_col=0); d1=pd.read_csv('out/proj2027_orig_d1.csv',index_col=0)
m=np.arange(1,13); names=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
ax.fill_between(m,d0.lo95,d0.hi95,color='#8e44ad',alpha=.15,label='95% prediction interval')
ax.fill_between(m,d0.lo80,d0.hi80,color='#8e44ad',alpha=.28,label='80% prediction interval')
ax.plot(m,d0['mean'],'o-',color='#5b2c6f',label='Stationary Fourier-SARIMA (point projection)')
ax.plot(m,d1['mean'],'s--',color='#555',ms=4,label='As-submitted model (d=1), point projection')
obs=y['2022':].groupby([y['2022':].index.year,y['2022':].index.month]).sum().unstack(0)
for yr,st in zip(obs.columns,[':','-.','--']): ax.plot(m,obs[yr],st,color='#2e86c1',lw=1,alpha=.8,label=f'Observed {yr}')
ax.set_xticks(m); ax.set_xticklabels(names); ax.set_ylabel('Monthly admissions'); ax.legend(fontsize=8,frameon=False,ncol=2,loc='upper left')
plt.tight_layout(); plt.savefig('out/Fig6_2027_scenario.png',dpi=600); plt.show()

## 11. Figure 1: study-area map
Boundaries: geoBoundaries thana/upazila level (CC BY 4.0; BBS/OCHA), via the bangladesh-geojson repository; country outlines and rivers: Natural Earth. The BMD Agargaon station is marked as the **assumed** source of the weather data.

In [ ]:
# Download boundary files (skipped if already present)
SRC = {'geo/bgd_adm3.geojson': 'https://raw.githubusercontent.com/ifahimreza/bangladesh-geojson/master/src/data/bangladesh.geojson',
       'geo/ne_50m_admin_0_countries.geojson': 'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_50m_admin_0_countries.geojson',
       'geo/ne_10m_rivers_lake_centerlines.geojson': 'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_rivers_lake_centerlines.geojson'}
for f, u in SRC.items():
    if not os.path.exists(f): urllib.request.urlretrieve(u, f)

import numpy as np, geopandas as gpd, pandas as pd, matplotlib.pyplot as plt
from shapely.geometry import box, Point
from matplotlib.patches import Patch, Rectangle, FancyArrow
from matplotlib.lines import Line2D
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9})
STUDY='#c44e52'; LINE='#4d4d4d'
g=gpd.read_file('geo/bgd_adm3.geojson')
cent=box(90.30,23.65,90.55,23.92); rp=g.representative_point()
metro=(g.district_name=='')&rp.within(cent)
metro|=g.source_name.isin(['Mirpur','Mohammadpur'])&rp.within(cent)
g.loc[metro,'district_name']='Dhaka'
fix={'Gazipur Sadar':'Gazipur','Kaliakair':'Gazipur','Manohardi':'Narsingdi','Narayanganj Sadar':'Narayanganj','Serajdikhan':'Munshiganj'}
for k,v in fix.items(): g.loc[(g.source_name==k)&(g.district_name==''),'district_name']=v
print('metropolitan thanas:',int(metro.sum()))
dist=g[g.district_name!=''].dissolve('district_name').reset_index()
dist['geometry']=dist.geometry.buffer(0.004).buffer(-0.004)
bgd=g.dissolve().geometry.iloc[0].buffer(0.01).buffer(-0.01)
study=g[metro].dissolve().geometry.iloc[0].buffer(0.002).buffer(-0.002)
ne=gpd.read_file('geo/ne_50m_admin_0_countries.geojson')
riv=gpd.read_file('geo/ne_10m_rivers_lake_centerlines.geojson')
stn=(90.3796,23.7782)   # BMD Dhaka station, Agargaon (approximate)

fig=plt.figure(figsize=(9.2,6.6))
ax=fig.add_axes([0.07,0.07,0.61,0.90])
x0,x1,y0,y1=89.98,90.72,23.48,24.05
lat0=np.deg2rad((y0+y1)/2)
dist[dist.district_name=='Dhaka'].plot(ax=ax,color='#ececec',lw=0,zorder=0.5)
g.boundary.plot(ax=ax,color='#bdbdbd',lw=0.35,zorder=1)
gpd.GeoSeries([study]).plot(ax=ax,color=STUDY,alpha=0.85,lw=0,zorder=2)
dist.boundary.plot(ax=ax,color=LINE,lw=1.0,zorder=3)
dist[dist.district_name=='Dhaka'].boundary.plot(ax=ax,color='black',lw=2.0,zorder=4)
riv.clip(box(x0,y0,x1,y1)).plot(ax=ax,color='#6baed6',lw=1.2,zorder=1.5)
ax.plot(*stn,marker='^',ms=11,mfc='#ffd92f',mec='black',mew=1.2,zorder=6,ls='none')
for nm,(lx,ly) in {'Gazipur':(90.50,23.99),'Manikganj':(90.06,23.78),'Narayanganj':(90.60,23.72),'Munshiganj':(90.38,23.53),'Narsingdi':(90.68,23.93)}.items():
    ax.text(lx,ly,nm,ha='center',va='center',fontsize=8.5,color='#333',style='italic',zorder=7)
ax.text(90.17,23.95,'Dhaka\ndistrict',ha='center',fontsize=8.5,style='italic',color='#111',zorder=7)
ax.set_xlim(x0,x1); ax.set_ylim(y0,y1); ax.set_aspect(1/np.cos(lat0))
ax.set_xticks(np.arange(90.0,90.71,0.2)); ax.set_yticks(np.arange(23.5,24.06,0.1))
ax.set_xticklabels([f'{v:.1f}°E' for v in ax.get_xticks()]); ax.set_yticklabels([f'{v:.1f}°N' for v in ax.get_yticks()])
ax.tick_params(labelsize=8); ax.set_xlabel(''); ax.set_ylabel('')
# scale bar 10 km
kmdeg=111.32*np.cos(lat0); sx,sy=x0+0.03,y0+0.025
for i,c in enumerate(['black','white']):
    ax.add_patch(Rectangle((sx+i*5/kmdeg,sy),5/kmdeg,0.008,fc=c,ec='black',lw=0.8,zorder=8))
for k in (0,5,10): ax.text(sx+k/kmdeg,sy+0.014,f'{k}',ha='center',fontsize=7.5,zorder=8)
ax.text(sx+10/kmdeg+0.012,sy+0.002,'km',fontsize=7.5,zorder=8)
ax.annotate('N',xy=(x1-0.04,y1-0.03),xytext=(x1-0.04,y1-0.10),ha='center',fontsize=11,fontweight='bold',
            arrowprops=dict(arrowstyle='-|>',color='black',lw=1.5),zorder=8)
ax.text(0.015,0.985,'B',transform=ax.transAxes,fontsize=14,fontweight='bold',va='top',zorder=9,bbox=dict(fc='white',ec='none',pad=1.5))
# inset
ai=fig.add_axes([0.70,0.52,0.29,0.45])
ne.plot(ax=ai,color='#f0f0f0',edgecolor='#9e9e9e',lw=0.4)
gpd.GeoSeries([bgd]).plot(ax=ai,color='#d9d9d9',edgecolor=LINE,lw=0.6)
gpd.GeoSeries([study]).plot(ax=ai,color=STUDY,lw=0)
ai.plot(90.40,23.78,'o',ms=6,mfc=STUDY,mec='black',mew=0.6)
ai.add_patch(Rectangle((x0,y0),x1-x0,y1-y0,fill=False,ec='black',lw=0.9))
ai.set_xlim(66,100); ai.set_ylim(5,37); ai.set_aspect(1/np.cos(np.deg2rad(21)))
for n,(lx,ly) in {'INDIA':(78.5,21.5),'NEPAL':(84,28.4),'CHINA':(88,33.5),'MYANMAR':(95.8,20),'PAKISTAN':(69.5,28.5),'BHUTAN':(90.6,28.3),'SRI LANKA':(80.7,9.2)}.items():
    ai.text(lx,ly,n,fontsize=6,ha='center',color='#555')
ai.text(97,8,'Bay of\nBengal',fontsize=6,ha='center',color='#4a7fb5',style='italic') if False else ai.text(88.5,15,'Bay of\nBengal',fontsize=6,ha='center',color='#4a7fb5',style='italic')
ai.annotate('BANGLADESH',xy=(90.0,24.3),xytext=(94.5,31.0),fontsize=6,ha='center',color='#222',arrowprops=dict(arrowstyle='-',color='#555',lw=0.6))
ai.set_xticks([]); ai.set_yticks([]); ai.set_xlabel(''); ai.set_ylabel(''); ai.set_facecolor('#eaf3fb')
ai.text(0.03,0.97,'A',transform=ai.transAxes,fontsize=14,fontweight='bold',va='top')
# legend
h=[Patch(fc=STUDY,alpha=0.85,label='Study area: Dhaka metropolitan\nthanas (DNCC + DSCC)'),
   Patch(fc='#ececec',ec='black',lw=2,label='Dhaka district'),
   Line2D([],[],color=LINE,lw=1,label='Other district boundaries'),
   Line2D([],[],color='#bdbdbd',lw=0.8,label='Thana / upazila boundaries'),
   Line2D([],[],color='#6baed6',lw=1.2,label='Major rivers'),
   Line2D([],[],marker='^',ms=10,mfc='#ffd92f',mec='black',ls='none',label='BMD Dhaka station, Agargaon\n(assumed source of weather data)'),
   Rectangle((0,0),1,1,fill=False,ec='black',label='Extent of panel B')]
fig.legend(handles=h,loc='lower left',bbox_to_anchor=(0.695,0.08),fontsize=8,frameon=False,handlelength=2.2,labelspacing=0.9)
fig.text(0.69,0.035,'Boundaries: geoBoundaries (CC BY 4.0), BBS/OCHA;\ncountries and rivers: Natural Earth.',fontsize=6.5,color='#555')
plt.savefig('out/Fig1_study_area.png',dpi=600); plt.savefig('out/Fig1_study_area.pdf'); plt.show()

## 12. Figure 2: study workflow
A plain version of the workflow. The figure used in the manuscript is a redesigned graphic of the same stages.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
plt.rcParams.update({'font.family':'DejaVu Sans'})
fig,ax=plt.subplots(figsize=(10,10.2)); ax.set_xlim(0,100); ax.set_ylim(14,115); ax.axis('off')
C={'data':'#e8eef6','prep':'#eef5ea','model':'#fdf1e3','eval':'#f6e9ef','long':'#ece9f6'}
def box(x,y,w,h,title,body,c):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.4,rounding_size=1.2',fc=c,ec='#555',lw=1))
    ax.text(x+w/2,y+h-1.6,title,ha='center',va='top',fontsize=9.5,fontweight='bold')
    ax.text(x+w/2,y+h-4.6,body,ha='center',va='top',fontsize=8,linespacing=1.35)
def arr(x1,y1,x2,y2):
    ax.annotate('',xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle='-|>',color='#333',lw=1.2))
box(22,103,56,10,'1. Data','Daily dengue hospital admissions and weather, Dhaka\n1 Jan 2020 – 30 Dec 2024 (n = 1,826 days; Kaggle)',C['data'])
arr(50,103,50,99.5)
box(22,88,56,11,'2. Data audit','Missing days (0), zero-count days (122), isolated spikes (9)\nComparison with published DGHS totals\nTwo training windows: from 2020 (full) and from 2022 (restricted)',C['prep'])
arr(36,88,24,82.5); arr(64,88,76,82.5)
box(2,67,44,15.5,'3a. Short-horizon forecasting','Rolling one-day-ahead forecasts\nPhase 1: train to 2023, test 2024 (typical year)\nPhase 2: train to 2022, test 2023 (outbreak year)\nTransformation, orders and hyperparameters\nchosen on training data only',C['model'])
box(54,70,44,12.5,'3b. Weather covariates','Granger tests: temperature, humidity, rainfall\n(weekly, deseasonalised, 4–12-week lags)\nOut-of-sample: SARIMA vs SARIMA + lagged\nweather (lag chosen on training data)',C['model'])
arr(24,67,24,61.5)
box(2,42,44,19.5,'4. Competing models','SARIMA (2,1,2)(2,0,2)₇ (order checked by AIC, 81 models)\nNaive benchmarks: persistence, seasonal naive,\n7-day mean\nML on a common feature set: Linear, Ridge, SVR, MLP,\nRandom Forest, XGBoost, LightGBM, CatBoost\n(walk-forward tuned); hybrid SVR + Linear Regression',C['model'])
arr(24,42,24,37.5)
box(2,18,44,19.5,'5. Evaluation','MAE, RMSE, sMAPE, MASE, R²\nDiebold–Mariano tests and block-bootstrap 95% CIs\nMetrics by transmission period\n(Jul–Nov vs Dec–Jun)\nResidual diagnostics (Ljung–Box)',C['eval'])
box(54,30,44,25,'6. Long-range scenario','Fourier-augmented SARIMA\n(annual harmonics; K chosen by AIC)\nHindcasts from end-2021, 2022, 2023\nvs climatological benchmark\n2027 monthly projection with\n80% and 95% prediction intervals\n(1,000 simulated paths)',C['long'])
arr(66,70,40,61.5)
arr(46.6,50,53.4,46)
ax.text(50,51.2,'SARIMA +\nFourier terms',fontsize=7,color='#555',ha='center',style='italic')
ax.text(66,63.5,'covariate result\ninforms model set',fontsize=7,color='#555',ha='center',style='italic')
plt.savefig('out/Fig2_workflow.png',dpi=600,bbox_inches='tight'); plt.savefig('out/Fig2_workflow.pdf',bbox_inches='tight'); plt.show()

## 13. Download all outputs

In [ ]:
import shutil
shutil.make_archive('revision_outputs', 'zip', 'out')
try:
    from google.colab import files; files.download('revision_outputs.zip')
except ImportError:
    print('Saved revision_outputs.zip')